# ForgeEdge — Alpha Discovery (Modulo 2)

Questo notebook illustra il modulo **AlphaDiscovery**, il terzo passo della pipeline FORGE, partendo da un dato letto da un file **Excel in locale**.

La catena è strettamente sequenziale:

```
Excel (OHLCV + KPI)
   │
   ▼
Market Context  (Modulo 0)   →  aggiunge la colonna 'regime'
   │
   ▼
Event Discovery (Modulo 1)   →  Event Candidates (condizioni booleane)
   │
   ▼
Alpha Discovery (Modulo 2)   →  Alpha Contract  ← questo notebook
```

## Cosa fa Alpha Discovery

Riceve gli Event Candidate da Event Discovery e, **per ogni evento**, _deriva dai dati_ il target economico — non lo riceve in input:

| Parametro derivato | Come |
|---|---|
| `holding_period_h` | orizzonte della griglia che massimizza `|z_h|`, l'**excess log-return** `Δ_h = μ_cond − μ_base` standardizzato da un **null a rotazione circolare** (autocorrelation-robust: una t-stat naïve si gonfierebbe sugli orizzonti lunghi per gli eventi clusterizzati, incollando `h*` al bordo della griglia anche senza edge reale) |
| `sell_pct` | quantile della **Maximum Favorable Excursion** (MFE) a `h*` sulle barre attive in-sample (`mfe_quantile`, default mediana, con floor) |
| `direction` | segno dell'excess log-return `Δ_h*` (long / short, automatico, mai imposto a priori) |

La significatività dell'excess — null a rotazione (`p_value_by_h`) + Benjamini-Hochberg (`h_sig`) — è **diagnostica non bloccante**: con pochi eventi il gate FDR può essere vuoto anche su un edge reale, quindi `h*` si sceglie comunque su `|z_h|` e il target è marcato `statistically_weak`.

Poiché scegliere l'orizzonte sulla griglia è un'ottimizzazione in-sample, la tabella è divisa temporalmente (`train_ratio`) e il target derivato viene **replicato out-of-sample** sulla coda mai usata nella derivazione. La conferma OOS — come IC, lift e Cohen's d — è una **diagnostica non bloccante** che alimenta il voto A–D del contratto: tutti i contratti con direzione determinata passano a Rule Discovery, l'unico giudice economico.

> **Nota sul dato.** Alpha Discovery **non ricalcola nulla**: legge gli eventi (`event_series`) e le feature dalla tabella post-pipeline di Event Discovery (`ed.df`), che porta già la colonna `regime` e tutte le feature derivate.

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, "../src")   # per esecuzione da notebooks/

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from forgedge import (
    MarketContext,
    EventDiscovery, DiscoveryConfig,
    AlphaDiscovery, AlphaConfig, PromotionThresholds,
)
from forgedge.event_discovery.models import GateParams

## 1. Caricamento da Excel in locale

Il dataset di esempio contiene candele orarie per uno o più simboli.
Filtriamo su un singolo simbolo, ordiniamo cronologicamente e costruiamo la colonna timestamp `open_dt`.

> Sostituisci `DATA_PATH` con il percorso al tuo file `.xlsx`.
> Il formato atteso è una riga per barra con almeno `open_time` (Unix ms) e `close`;
> le altre colonne OHLCV / indicatori vengono usate da Event Discovery per generare le feature.


In [ ]:
# ── Percorso al file Excel locale ──────────────────────────────────────────
DATA_PATH = "../data/test1h.xlsx"
SYMBOL    = "ADAUSDC"

df_raw = pd.read_excel(DATA_PATH)
df = df_raw[df_raw["symbol"] == SYMBOL].copy().sort_values("open_time")

# Timestamp leggibile (open_time è in millisecondi Unix)
df["open_dt"] = pd.to_datetime(df["open_time"], unit="ms")
df = df.drop(columns=[c for c in ("symbol", "timeframe", "open_time") if c in df.columns])

print(f"Simbolo : {SYMBOL}")
print(f"Righe   : {len(df):,}")
print(f"Periodo : {df['open_dt'].iloc[0].date()} → {df['open_dt'].iloc[-1].date()}")
print(f"Colonne : {list(df.columns)}")
df.head(3)

## 2. Modulo 0 — Market Context

`MarketContext` etichetta ogni barra con un `regime` (e `regime_stable`).
Alpha Discovery legge questa colonna per la **regime sensitivity** — non la ricalcola.


In [ ]:
mc = MarketContext(df.copy())
enriched = mc.run()

print("Distribuzione dei regimi:")
print(mc.distribution())

## 3. Modulo 1 — Event Discovery

`EventDiscovery` genera gli Event Candidate (condizioni booleane sulle feature) **senza guardare il forward return**.
Passiamo la tabella già arricchita di `regime`: in questo modo `ed.df` — la tabella post-pipeline — porta sia il regime sia tutte le feature derivate, pronta per Alpha Discovery.


In [ ]:
ed = EventDiscovery(
    enriched.copy(),
    DiscoveryConfig(
        timestamp_col="open_dt",
        gate_params=GateParams(min_act=50, min_months=8, max_conc=0.40, min_tpm=2.0),
        max_and_components=2,  # composizione single-pass legacy; il default della classe e' 1 dall'issue #254 Fase 8
    ),
)
candidates = ed.run()

print(f"Event Candidates : {len(candidates)}")
print(f"  singoli        : {sum(1 for c in candidates if len(c.components) == 1)}")
print(f"  AND            : {sum(1 for c in candidates if len(c.components) > 1)}")

## 4. Modulo 2 — Alpha Discovery

`AlphaConfig` **non contiene un target economico**: il target è derivato per evento.
I parametri che governano la derivazione e la promozione sono:

| Parametro | Default | Ruolo |
|---|---|---|
| `horizon_grid` | `(1, 2, …, 48)` | orizzonti candidati (in barre) scanditi per derivare `h*` |
| `train_ratio` | `0.7` | quota in-sample (derivazione + misure); la coda resta per la conferma OOS |
| `thresholds.min_lift` | `0.08` | lift minimo sopra il base rate |
| `thresholds.min_cohens_d` | `0.15` | effect size minimo |
| `thresholds.min_activations` | `30` | base statistica minima |
| `thresholds.use_fdr` / `fdr_q` | `True` / `0.10` | controllo del False Discovery Rate (Benjamini-Hochberg) |
| `thresholds.oos_max_p` | `0.10` | p-value massimo per la conferma OOS del target derivato |
| `thresholds.min_oos_activations` | `10` | attivazioni OOS minime perché la conferma sia valutata |

`asset` / `exchange` / `timeframe` / `fee_per_side` sono **solo metadati di tracciabilità**: vengono copiati nel contratto ma non entrano in alcuna misura.


In [ ]:
config = AlphaConfig(
    # Derivazione del target — griglia di orizzonti e split IS/OOS
    horizon_grid=(1, 2, 3, 4, 6, 8, 12, 16, 24, 36, 48),
    train_ratio=0.7,
    mfe_quantile=0.5,   # quantile MFE per il sell_pct (0.5 = mediana)
    mfe_floor=0.005,    # floor del sell_pct (50 bp)
    # Metadati di tracciabilità (nessun ruolo nel calcolo)
    asset=SYMBOL,
    exchange="binance_spot",
    timeframe="1H",
    thresholds=PromotionThresholds(
        min_lift=0.08,
        min_cohens_d=0.15,
        min_activations=30,
        use_fdr=True,
        fdr_q=0.10,
        oos_max_p=0.10,
        min_oos_activations=10,
    ),
)

# ed.df è la tabella post-pipeline: regime + feature derivate + event_series.
# Alpha Discovery legge da lì — niente viene ricalcolato.
ad = AlphaDiscovery(ed.df, candidates, config)
contracts = ad.run()

promoted = ad.promoted_contracts()
print(f"Contratti valutati    : {len(contracts)}")
print(f"Promossi (HYPOTHESIS) : {len(promoted)}")   # tutti i contratti con direzione determinata
from collections import Counter
print(f"Distribuzione voti    : {dict(Counter(c.alpha_score.grade for c in promoted))}")

## 5. Struttura di mercato e split IS/OOS

Lo Step 2 calcola — sulla sola finestra in-sample — l'esponente di Hurst e il profilo di autocorrelazione: il **contesto interpretativo** per leggere i risultati (un alpha mean-reversion è atteso solo se il mercato è mean-reverting, `H < 0.5`).


In [ ]:
ms = ad.market_structure
n = len(ed.df)

print("── Struttura di mercato (Step 2, in-sample) ──")
print(f"Hurst        : {ms.hurst:.3f}  ({ms.hurst_interpretation})")
print(f"Famiglia att.: {ms.expected_family}")
print("ACF return   : " + "  ".join(f"lag{l}={v:+.3f}" for l, v in ms.autocorr.items()))
print()
print("── Split temporale ──")
print(f"In-sample  : barre 0 … {ad.split_idx-1}  ({ad.split_idx} barre, "
      f"{ad.split_idx/n:.0%})")
print(f"Out-of-sample: barre {ad.split_idx} … {n-1}  ({n - ad.split_idx} barre, "
      f"{(n-ad.split_idx)/n:.0%})")

## 6. Summary dei candidati valutati

`ad.summary()` restituisce un DataFrame piatto, ordinato per `composite_score` decrescente.
Include i parametri **derivati** (`holding_period_h`, `sell_pct`, `direction`) e l'esito della conferma OOS (`oos_passed`).


In [ ]:
summary = ad.summary()

cols = [
    "expression", "feature", "holding_period_h", "sell_pct", "direction",
    "ic", "n_activations", "win_rate", "base_rate", "lift", "cohens_d",
    "p_value", "oos_passed", "regime_dependency", "composite_score", "grade",
]
summary[cols].head(12)

### I contratti per priorità (top 10)

Tutti i contratti con direzione determinata sono promossi (`status = "HYPOTHESIS"`) e passano a Rule Discovery. Le misure statistiche — admission IC, lift, effect size, FDR e replay OOS — alimentano il **voto A–D** e le eventuali note `[diagnostic]`, ma non scartano il candidato: Rule Discovery è l'unico giudice economico.


In [ ]:
promoted_view = summary[summary["promoted"]].head(10)[cols]
print(f"Alpha promossi: {len(summary[summary['promoted']])}")
promoted_view

## 7. Il target derivato di un singolo evento

Per il candidato migliore, ispezioniamo **come** il target è stato derivato: il profilo dell'excess log-return `Δ_h` e dello score di selezione `|z_h|` lungo tutta la griglia di orizzonti. L'orizzonte scelto (`h*`) è quello che massimizza `|z_h|` (ristretto a `h_sig` se non vuoto); il `sell_pct` è il quantile della MFE a `h*` sulle barre attive IS.

In [ ]:
if promoted:
    best = max(promoted, key=lambda c: c.alpha_score.composite_score)
else:
    # fallback: il candidato col composite score più alto, anche se non promosso
    best = max(contracts, key=lambda c: c.alpha_score.composite_score)

dt = best.derived_target

print(f"event_candidate_id : {best.event_candidate_id}")
print(f"expression         : {best.event_expression}")
print()
print(f"── Target derivato ──")
print(f"holding_period_h*  : {dt.holding_period_h}  (argmax |z_h|, excess standardizzato dal null a rotazione)")
print(f"sell_pct           : {dt.sell_pct:.4f}  (quantile MFE a h*, barre attive IS)")
print(f"direction          : {dt.direction}  (segno dell'excess Δ_h*)")
print(f"mean_advantage     : {dt.mean_advantage:+.5f}  (Δ_h* — excess log-return firmato)")
print(f"h_sig              : {list(dt.h_sig)}  (orizzonti che superano BH — diagnostica)")
print(f"statistically_weak : {dt.statistically_weak}")

# Profilo lungo la griglia
profile = pd.DataFrame({
    "horizon_h":       list(dt.advantage_by_h.keys()),
    "excess_logret":   list(dt.advantage_by_h.values()),   # Δ_h
    "z_stat":          [dt.t_stat_by_h[h] for h in dt.advantage_by_h],   # z_h (rotazione)
    "score_absZ":      [dt.score_by_h[h] for h in dt.advantage_by_h],    # |z_h|
    "p_value":         [dt.p_value_by_h[h] for h in dt.advantage_by_h],  # rotazione
})
profile["in_h_sig"]  = profile["horizon_h"].isin(dt.h_sig)
profile["selected"]  = profile["horizon_h"] == dt.holding_period_h
profile

In [ ]:
# Visualizzazione del profilo di derivazione
fig, ax1 = plt.subplots(figsize=(12, 4))

ax1.bar(profile["horizon_h"].astype(str), profile["excess_logret"],
        color=np.where(profile["selected"], "darkorange", "steelblue"),
        alpha=0.85, label="excess log-return Δ_h")
ax1.axhline(0, color="gray", lw=0.8)
ax1.set_xlabel("orizzonte (barre)")
ax1.set_ylabel("Δ_h (excess log-return, firmato)", color="steelblue")
ax1.set_title(f"Derivazione del target — {best.event_expression}")

ax2 = ax1.twinx()
ax2.plot(profile["horizon_h"].astype(str), profile["score_absZ"],
         color="crimson", marker="o", lw=1.5, label="score = |z_h|")
ax2.set_ylabel("score = |z_h|  (excess standardizzato, null a rotazione)", color="crimson")

h_star_pos = profile.index[profile["selected"]][0]
ax2.annotate("h*  (max |z_h|)",
             xy=(h_star_pos, profile.loc[h_star_pos, "score_absZ"]),
             xytext=(h_star_pos, profile["score_absZ"].max() * 0.9),
             ha="center", color="crimson", fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Replay out-of-sample

Il blocco `oos_validation` replica il target derivato sulla coda temporale **mai usata** nella derivazione né in alcuna misura in-sample. `passed = True` quando:

- almeno `min_oos_activations` attivazioni OOS,
- vantaggio medio orientato ancora positivo (il segno si conferma),
- t-test one-sided sotto `oos_max_p`.

Una conferma mancata **non blocca** il contratto: viene registrata come nota `[diagnostic]` e pesa sul voto A–D.


In [ ]:
oos = best.oos_validation
if oos is None:
    print("Nessuna validazione OOS (train_ratio = 1.0).")
else:
    print(f"── Conferma OOS del target derivato ──")
    print(f"barre OOS          : {oos.n_bars}")
    print(f"attivazioni OOS    : {oos.n_activations}")
    print(f"mean_advantage OOS : {oos.mean_advantage:+.5f}  (orientato: >0 = favorevole)")
    print(f"t_stat / p_value   : {oos.t_stat:.2f} / {oos.p_value:.4f}")
    print(f"win_rate / base    : {oos.win_rate:.1%} / {oos.base_rate:.1%}  (lift {oos.lift:+.1%})")
    print(f"passed             : {oos.passed}")

### Confronto IS vs OOS

Mettere a confronto le misure in-sample con quelle out-of-sample è il modo più diretto per vedere quanto il target derivato "regge" fuori campione (un degrado moderato è fisiologico; un'inversione di segno è un campanello d'allarme).


In [ ]:
es = best.event_stats
if oos is not None:
    cmp = pd.DataFrame({
        "in_sample":     [es.win_rate, es.base_rate, es.lift, es.fwd_return_mean, es.p_value],
        "out_of_sample": [oos.win_rate, oos.base_rate, oos.lift, oos.mean_advantage, oos.p_value],
    }, index=["win_rate", "base_rate", "lift", "mean_advantage", "p_value"])
    display(cmp.round(4))

## 9. Regime sensitivity

Lo Step 5 misura IC e win rate **per regime** (usando la colonna `regime` di Market Context) e classifica la dipendenza dell'alpha:

- **agnostic** — significativo in tutti i regimi → deploy diretto
- **conditional** — significativo in alcuni → deploy con filtro di regime
- **specific** — significativo in un solo regime
- **broken** — mai significativo


In [ ]:
ra = best.regime_analysis
print(f"Tipo di dipendenza : {ra.dependency_type}")
print(f"Regime breadth     : {ra.regime_breadth:.2f}")
print(f"Regimi attivi      : {ra.active_regimes}")
print(f"Regimi deboli      : {ra.weak_regimes}")
print()

if ra.per_regime:
    reg_df = pd.DataFrame([{
        "regime":   r.regime,
        "n":        r.n,
        "ic":       r.ic,
        "p_value":  r.p_value,
        "win_rate": r.win_rate,
        "strength": r.strength,
    } for r in ra.per_regime])
    display(reg_df.round(4))
else:
    print("Nessuna colonna 'regime' nella tabella → regime sensitivity non valutata.")

## 10. L'Alpha Contract completo

`to_contract_dict()` serializza il contratto nella struttura annidata documentata (Sezione 2 di `AlphaDiscovery.md`), pronta per YAML/JSON. È l'**interfaccia formale** verso Rule Discovery.


In [ ]:
import json as _json

contract_dict = best.to_contract_dict()

# Stampa le sezioni principali (il dict completo è annidato)
for section in ("alpha_id", "status", "direction", "event_candidate_id",
                "event_expression", "pattern_family"):
    print(f"{section:20s}: {contract_dict[section]}")

print("\nderived_target:")
print(_json.dumps(contract_dict["derived_target"], indent=2, default=str))

print("\noos_validation:")
print(_json.dumps(contract_dict["oos_validation"], indent=2, default=str))

In [ ]:
# Esportazione su file YAML (opzionale — richiede pyyaml)
try:
    import yaml
    out_path = f"{best.alpha_id}.yaml"
    with open(out_path, "w") as fh:
        yaml.safe_dump(contract_dict, fh, sort_keys=False, allow_unicode=True)
    print(f"Contratto scritto in: {out_path}")
except ImportError:
    print("pyyaml non installato — salto l'esportazione YAML "
          "(il dict di to_contract_dict() è comunque pronto per la serializzazione).")

## 11. Alpha Score e griglia di prioritizzazione

Il `composite_score` (combinazione pesata di IC, lift, Cohen's d e regime breadth) ordina i contratti; il `grade` (A ≥ 0.75 / B ≥ 0.50 / C ≥ 0.25 / D) li raggruppa per priorità. Il voto **non è un gate**: tutti i contratti A–D passano a Rule Discovery, che li giudica economicamente in ordine di priorità.


In [ ]:
score_view = (
    summary[summary["promoted"]]
    [["expression", "holding_period_h", "direction", "ic", "lift",
      "cohens_d", "regime_breadth", "composite_score", "grade"]]
    .head(15)
)
display(score_view)

# Distribuzione dei grade tra i promossi
if len(summary[summary["promoted"]]):
    grade_counts = summary[summary["promoted"]]["grade"].value_counts()
    fig, ax = plt.subplots(figsize=(7, 3.5))
    grade_counts.reindex(["A", "B", "C", "D"]).dropna().plot(
        kind="bar", color="seagreen", ax=ax, width=0.7)
    ax.set_title("Distribuzione dei grade — Alpha promossi")
    ax.set_xlabel("grade"); ax.set_ylabel("n. alpha")
    plt.tight_layout(); plt.show()

## 12. Direzioni derivate — long vs short

Poiché la direzione è derivata dal **segno dell'excess log-return** (non dal rendimento condizionato grezzo, che il drift dell'asset distorce), in un singolo run possono emergere sia alpha long sia short. Qui contiamo come si distribuiscono tra i candidati promossi.

In [ ]:
prom = summary[summary["promoted"]]
if len(prom):
    by_dir = prom.groupby("direction").agg(
        n=("expression", "count"),
        mean_lift=("lift", "mean"),
        mean_score=("composite_score", "mean"),
        mean_h=("holding_period_h", "mean"),
    ).round(3)
    display(by_dir)
else:
    print("Nessun alpha promosso con i parametri correnti.")

## 13. Riepilogo

Partendo da un Excel locale abbiamo eseguito l'intera catena FORGE fino all'Alpha Contract:

1. **Excel → DataFrame** — un simbolo, ordinato cronologicamente, con `open_dt`.
2. **Market Context** — colonna `regime` per ogni barra.
3. **Event Discovery** — Event Candidate booleani; `ed.df` porta regime + feature derivate.
4. **Alpha Discovery** —
   - per ogni evento **deriva** `(h*, sell_pct, direction)`: h* = argmax `|z_h|` (excess log-return `Δ_h = μ_cond − μ_base` standardizzato da un null a rotazione circolare), direction = segno di `Δ_h*`, sell_pct = quantile MFE a h*;
   - misura IC, win rate, lift, Cohen's d, regime sensitivity al target derivato;
   - controlla i falsi positivi multipli con Benjamini-Hochberg;
   - **replica il target derivato out-of-sample** e assegna a ogni contratto un voto A–D (le misure sono diagnostiche, non gate);
   - compila un **Alpha Contract** per ogni candidato.

> **Nota chiave.** Alpha Discovery non riceve parametri economici e non ricalcola eventi o feature: legge gli output dei moduli a monte e _deriva_ il target dai dati, misurandolo anche fuori campione. Tutti i contratti `HYPOTHESIS` (voti A–D) sono l'input di **Rule Discovery** (Modulo 3), l'unico giudice economico, che ne verifica l'eseguibilità con la meccanica reale degli ordini.